# Activation Files — Dimension Inspection

Loads all `.pt` files from `data/activations/` for the three harm categories:
- **cybercrime** (`cybercrime_cone.pt`, `cybercrime_seed_dir.pt`)
- **selfharm** (`selfharm_cone.pt`, `selfharm_seed_dir.pt`)
- **violence** (`violence_cone.pt`, `violence_seed_dir.pt`)

For each file the notebook reports: shape, dtype, device, min/max/mean values, and memory footprint.

In [1]:
import json
import os
from pathlib import Path

import torch
import pandas as pd

pd.set_option("display.max_colwidth", None)

## 1 — Paths & metadata

In [2]:
# Resolve paths relative to this notebook's location
NOTEBOOK_DIR = Path(os.path.abspath(''))
REPO_ROOT    = NOTEBOOK_DIR.parent          # one level up from phase1/
ACT_DIR      = REPO_ROOT / 'data' / 'activations'

print(f'Repo root   : {REPO_ROOT}')
print(f'Activations : {ACT_DIR}')
assert ACT_DIR.exists(), f'Directory not found: {ACT_DIR}'

# Load the experiment metadata
with open(ACT_DIR / 'metadata.json') as f:
    meta = json.load(f)

print('\nmetadata.json:')
print(json.dumps(meta, indent=2))

Repo root   : /home/sajib/Documents/Geometric-Auditing-Framework
Activations : /home/sajib/Documents/Geometric-Auditing-Framework/data/activations

metadata.json:
{
  "model": "/home/samuel/research/llmattacks/llm-attacks/DIR/Llama-3.1-8B-Instruct",
  "layer": 16,
  "cone_dim": 3,
  "spec_lambda": 0.5,
  "hidden_size": 4096,
  "categories": [
    "violence",
    "cybercrime",
    "selfharm"
  ]
}


## 2 — Enumerate .pt files

In [3]:
CATEGORIES = ['cybercrime', 'selfharm', 'violence']
FILE_TYPES  = ['cone', 'seed_dir']

pt_files = {}
for cat in CATEGORIES:
    for ft in FILE_TYPES:
        key  = f'{cat}_{ft}'
        path = ACT_DIR / f'{key}.pt'
        if path.exists():
            pt_files[key] = path
        else:
            print(f'[WARNING] Not found: {path}')

print(f'Found {len(pt_files)} .pt file(s):')
for k, p in pt_files.items():
    size_kb = p.stat().st_size / 1024
    print(f'  {k:<30}  {size_kb:>8.1f} KB')

Found 6 .pt file(s):
  cybercrime_cone                     49.2 KB
  cybercrime_seed_dir                  9.2 KB
  selfharm_cone                       49.2 KB
  selfharm_seed_dir                    9.2 KB
  violence_cone                       49.2 KB
  violence_seed_dir                    9.2 KB


## 3 — Load tensors & inspect dimensions

In [4]:
records = []

for key, path in pt_files.items():
    obj = torch.load(path, map_location='cpu', weights_only=True)
    
    # obj could be a raw Tensor or a dict/list wrapping tensors
    if isinstance(obj, torch.Tensor):
        tensor = obj
    elif isinstance(obj, dict):
        # If it is a dict, show keys and pick first tensor for stats
        print(f'\n{key} is a dict with keys: {list(obj.keys())}')
        tensor = next(v for v in obj.values() if isinstance(v, torch.Tensor))
    else:
        print(f'\n{key} has unexpected type: {type(obj)}')
        tensor = None

    if tensor is not None:
        records.append({
            'file'       : f'{key}.pt',
            'category'   : key.split('_')[0],
            'type'       : '_'.join(key.split('_')[1:]),
            'shape'      : tuple(tensor.shape),
            'ndim'       : tensor.ndim,
            'dtype'      : str(tensor.dtype),
            'device'     : str(tensor.device),
            'min'        : round(tensor.min().item(), 6),
            'max'        : round(tensor.max().item(), 6),
            'mean'       : round(tensor.mean().item(), 6),
            'std'        : round(tensor.std().item(), 6),
            'mem_MB'     : round(tensor.element_size() * tensor.nelement() / 1e6, 4),
        })

df = pd.DataFrame(records)
print('\nAll .pt tensors loaded successfully.\n')
df


All .pt tensors loaded successfully.



,file,category,type,shape,ndim,dtype,device,min,max,mean,std,mem_MB
0,cybercrime_cone.pt,cybercrime,cone,"(3, 4096)",2,torch.float32,cpu,-0.045898,0.045138,-0.000258,0.015624,0.0492
1,cybercrime_seed_dir.pt,cybercrime,seed_dir,"(4096,)",1,torch.bfloat16,cpu,-0.296875,0.096680,-0.000122,0.015625,0.0082
2,selfharm_cone.pt,selfharm,cone,"(3, 4096)",2,torch.float32,cpu,-0.172662,0.045536,0.000133,0.015625,0.0492
3,selfharm_seed_dir.pt,selfharm,seed_dir,"(4096,)",1,torch.bfloat16,cpu,-0.380859,0.120605,-0.000114,0.015625,0.0082
4,violence_cone.pt,violence,cone,"(3, 4096)",2,torch.float32,cpu,-0.046107,0.044390,-0.000202,0.015624,0.0492
5,violence_seed_dir.pt,violence,seed_dir,"(4096,)",1,torch.bfloat16,cpu,-0.225586,0.079102,-0.000117,0.015625,0.0082


## 4 — Per-category deep dive

In [5]:
for cat in CATEGORIES:
    print(f"{'='*60}")
    print(f"  CATEGORY: {cat.upper()}")
    print(f"{'='*60}")
    subset = df[df['category'] == cat][['file', 'shape', 'dtype', 'min', 'max', 'mean', 'std', 'mem_MB']]
    print(subset.to_string(index=False))
    print()


  CATEGORY: CYBERCRIME
                  file     shape          dtype       min      max      mean      std  mem_MB
    cybercrime_cone.pt (3, 4096)  torch.float32 -0.045898 0.045138 -0.000258 0.015624  0.0492
cybercrime_seed_dir.pt   (4096,) torch.bfloat16 -0.296875 0.096680 -0.000122 0.015625  0.0082

  CATEGORY: SELFHARM
                file     shape          dtype       min      max      mean      std  mem_MB
    selfharm_cone.pt (3, 4096)  torch.float32 -0.172662 0.045536  0.000133 0.015625  0.0492
selfharm_seed_dir.pt   (4096,) torch.bfloat16 -0.380859 0.120605 -0.000114 0.015625  0.0082

  CATEGORY: VIOLENCE
                file     shape          dtype       min      max      mean      std  mem_MB
    violence_cone.pt (3, 4096)  torch.float32 -0.046107 0.044390 -0.000202 0.015624  0.0492
violence_seed_dir.pt   (4096,) torch.bfloat16 -0.225586 0.079102 -0.000117 0.015625  0.0082



## 5 — Shape consistency check

Verify that the same file-type shares identical dimensions across categories (e.g., all `*_cone.pt` tensors have the same shape).

In [6]:
for ft in FILE_TYPES:
    subset = df[df['type'] == ft]
    shapes = subset['shape'].unique()
    consistent = len(shapes) == 1
    status = '✅ consistent' if consistent else '⚠️  INCONSISTENT'
    print(f'{ft:<12}  {status}  shapes={shapes}')

cone          ✅ consistent  shapes=[(3, 4096)]
seed_dir      ✅ consistent  shapes=[(4096,)]


## 6 — Cross-reference with metadata

In [7]:
expected_hidden = meta.get('hidden_size', None)
expected_cone   = meta.get('cone_dim', None)
layer           = meta.get('layer', None)

print(f'Expected from metadata.json:')
print(f'  hidden_size : {expected_hidden}')
print(f'  cone_dim    : {expected_cone}')
print(f'  layer       : {layer}')
print()

for _, row in df.iterrows():
    shape = row['shape']
    # For seed_dir we expect shape[-1] == hidden_size
    # For cone we expect shape[0] == cone_dim and shape[-1] == hidden_size (or similar)
    hidden_ok = (expected_hidden is not None) and (expected_hidden in shape)
    flag = '✅' if hidden_ok else '⚠️ '
    print(f'{flag} {row["file"]:<35}  shape={shape}  hidden_match={hidden_ok}')

Expected from metadata.json:
  hidden_size : 4096
  cone_dim    : 3
  layer       : 16

✅ cybercrime_cone.pt                   shape=(3, 4096)  hidden_match=True
✅ cybercrime_seed_dir.pt               shape=(4096,)  hidden_match=True
✅ selfharm_cone.pt                     shape=(3, 4096)  hidden_match=True
✅ selfharm_seed_dir.pt                 shape=(4096,)  hidden_match=True
✅ violence_cone.pt                     shape=(3, 4096)  hidden_match=True
✅ violence_seed_dir.pt                 shape=(4096,)  hidden_match=True
